In [10]:
import numpy as np
from Bio import Phylo
import sys
sys.path.append('../../pysimARG')
from clonal_genealogy import ClonalTree
from ClonalOrigin_ARG import ARG
from add_mutation import add_mutation
from localtree import LocalTree
from newick_to_tree import newick_to_tree

## Local tree selection

In [34]:
import re
import io
from Bio import Phylo

def load_local_trees(local_tree_file):
    genome_blocks = []

    # ^\[(\d+)\]  --> Matches [number] at the start of the line and captures the number
    # (.*)        --> Captures everything else (the Newick string)
    pattern = re.compile(r'^\[(\d+)\](.*)')
    
    print("Parsing local trees...")
    
    with open(local_tree_file, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
                
            match = pattern.match(line)
            if match:
                span_length = int(match.group(1))

                newick_str = match.group(2)

                tree_obj = Phylo.read(io.StringIO(newick_str), "newick")

                genome_blocks.append((span_length, tree_obj))
                
    return genome_blocks

In [35]:
def get_tree_at_position(genome_blocks, target_site):
    """
    Finds the local tree that covers a specific site on the genome.
    
    Parameters:
    - genome_blocks: List of tuples (span_length, tree_object)
    - target_site: Integer representing the genomic position (1-based indexing)
    
    Returns:
    - tree_object, block_start, block_end
    """
    if target_site < 1:
        print("Error: Target site must be 1 or greater.")
        return None, None, None

    current_start = 1
    
    for span, tree in genome_blocks:

        current_end = current_start + span - 1

        if current_start <= target_site <= current_end:
            return tree, current_start, current_end

        current_start += span

    print(f"Error: Site {target_site} is beyond the simulated genome length (Max: {current_start - 1}).")
    return None, None, None

In [4]:
file_path = "../../data/SimBac/local_tree.nwk" # Replace with your actual file name

blocks = load_local_trees(file_path)

print(f"\nSuccessfully loaded {len(blocks)} local trees.")

# Let's inspect the first few blocks to see where they map on the genome
print("\n--- Genome Map Overview ---")
current_position = 1

for i, (span, tree) in enumerate(blocks[:5]): # Just looking at the first 5
    end_position = current_position + span - 1
    print(f"Block {i+1}: Sites {current_position:05d} to {end_position:05d} (Length: {span}bp) - Tree has {len(tree.get_terminals())} leaves")
    current_position += span

Parsing local trees...

Successfully loaded 10618 local trees.

--- Genome Map Overview ---
Block 1: Sites 00001 to 00025 (Length: 25bp) - Tree has 10 leaves
Block 2: Sites 00026 to 00060 (Length: 35bp) - Tree has 10 leaves
Block 3: Sites 00061 to 00088 (Length: 28bp) - Tree has 10 leaves
Block 4: Sites 00089 to 00119 (Length: 31bp) - Tree has 10 leaves
Block 5: Sites 00120 to 00123 (Length: 4bp) - Tree has 10 leaves


In [5]:
# mock_blocks = [
#     (25, "Tree_A"), # Covers sites 1 to 25
#     (35, "Tree_B"), # Covers sites 26 to 60
#     (28, "Tree_C")  # Covers sites 61 to 88
# ]

# Let's look for site 40
target_position = 119

tree, start, end = get_tree_at_position(blocks, target_position)

In [6]:
tree, start, end

(Tree(rooted=False, weight=1.0), 89, 119)

In [7]:
Phylo.draw_ascii(tree)

                                                       ___________________ 4
  ____________________________________________________|
 |                                                    |                ___ 2
 |                                                    |_______________|
 |                                                                    |___ 8
 |
 |                                                         _______________ 9
_|                                                        |
 |                                                        |              , 6
 |                                                 _______|   ___________|
 |                                                |       | ,|           | 7
 |                                                |       | ||
 |                                                |       |_||____________ 3
 |________________________________________________|         |
                                                  |         |_________

In [8]:
target_position = 120

tree, start, end = get_tree_at_position(blocks, target_position)
tree, start, end

(Tree(rooted=False, weight=1.0), 120, 123)

In [9]:
Phylo.draw_ascii(tree)

                                                           _______________ 4
  ________________________________________________________|
 |                                                        |             ___ 2
 |                                                        |____________|
 |                                                                     |___ 8
 |
 |                                                             ___________ 9
_|                                                            |
 |                                                            |          , 6
 |                                                      ______|  ________|
 |                                                     |      |,|        | 7
 |                                                     |      |||
 |                                                     |      |||_________ 3
 |_____________________________________________________|       |
                                                     

## Review seq homoplasy

In [12]:
np.random.seed(100)
tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
tree.edge = edge
tree.node_height = node_height
tree.height = np.max(node_height)
tree.length = np.sum(edge[:, 2])

rho_site = 0.02
theta_site = 0.05
L = 2000
delta = 30

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

In [13]:
ARG_sim = ARG(tree, rho_site, L, delta, L, "seq")
node_site = add_mutation(ARG_sim, theta_site)

In [16]:
n_leaf = ARG_sim.n
n_site = ARG_sim.node_mat.shape[1]
m_vec = np.zeros(n_site, dtype=int)
s_vec = np.zeros(n_site, dtype=int)

In [17]:
n_leaf, n_site, m_vec, s_vec

(10,
 2000,
 array([0, 0, 0, ..., 0, 0, 0], shape=(2000,)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2000,)))

In [21]:
arg_site = LocalTree(ARG_sim, 0)
node_vec = arg_site.node_index.astype(int)
node_vec

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  13,  15,
        17,  18,  20,  22,  24,  26,  28,  30,  32,  34,  36,  37,  39,
        41,  43,  45,  46,  48,  49,  51,  53,  54,  55,  56,  58,  60,
        62,  63,  65,  67,  69,  71,  73,  75,  76,  77,  78,  80,  82,
        84,  85,  86,  87,  88,  90,  91,  92,  93,  95,  97,  99, 100,
       102, 103, 105, 106, 107, 108, 109, 110, 112, 114, 116, 117, 119,
       120, 122, 123, 125, 126, 127, 128, 130, 131, 132, 134, 135, 137,
       139, 140, 141, 142, 144, 146, 147, 148, 150, 151, 152, 154, 155,
       157, 159, 161, 163, 165, 167, 168, 169, 171, 173, 175, 176, 177,
       178, 180, 182, 184, 186, 188, 189, 190, 191, 193, 195, 197, 199,
       200, 202, 204, 206, 207, 209, 210, 212, 213, 214, 215, 216, 218,
       219, 221, 223, 224, 226, 227, 229, 230, 231, 232, 233, 234, 236,
       238, 240, 242, 243, 244, 245, 247, 248, 249, 251, 252, 253, 255,
       257, 259])

In [23]:
arg_site.node_index

array([  1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,  11.,
        13.,  15.,  17.,  18.,  20.,  22.,  24.,  26.,  28.,  30.,  32.,
        34.,  36.,  37.,  39.,  41.,  43.,  45.,  46.,  48.,  49.,  51.,
        53.,  54.,  55.,  56.,  58.,  60.,  62.,  63.,  65.,  67.,  69.,
        71.,  73.,  75.,  76.,  77.,  78.,  80.,  82.,  84.,  85.,  86.,
        87.,  88.,  90.,  91.,  92.,  93.,  95.,  97.,  99., 100., 102.,
       103., 105., 106., 107., 108., 109., 110., 112., 114., 116., 117.,
       119., 120., 122., 123., 125., 126., 127., 128., 130., 131., 132.,
       134., 135., 137., 139., 140., 141., 142., 144., 146., 147., 148.,
       150., 151., 152., 154., 155., 157., 159., 161., 163., 165., 167.,
       168., 169., 171., 173., 175., 176., 177., 178., 180., 182., 184.,
       186., 188., 189., 190., 191., 193., 195., 197., 199., 200., 202.,
       204., 206., 207., 209., 210., 212., 213., 214., 215., 216., 218.,
       219., 221., 223., 224., 226., 227., 229., 23

In [24]:
node_site_vec = node_site[node_vec - 1, 0]
node_site_vec

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [18]:
for site_loc in range(n_site):
    # Get local tree at this site
    arg_site = LocalTree(ARG_sim, site_loc)
    node_vec = arg_site.node_index.astype(int)
    node_site_vec = node_site[node_vec - 1, site_loc]  # Convert to 0-indexed

    # Compute minimum possible changes
    leaf_states = node_site_vec[:n_leaf]
    if np.any(leaf_states) and not np.all(leaf_states):
        m_vec[site_loc] = 1

    # Compute actual changes using Fitch algorithm
    s_site = 0
    site_dict = {}

    # Initialize leaf nodes
    site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

    # Process internal nodes
    for i in range(n_leaf, len(node_vec)):
        parent_node = node_vec[i]
        node_indices = np.where(arg_site.edge[:, 0] == parent_node)[0]
        if len(node_indices) == 2:
            # Coalescent structure (two children)
            children_node = arg_site.edge[node_indices, 1].astype(int)
            child_1_states = site_dict[children_node[0]]
            child_2_states = site_dict[children_node[1]]

            intersec = child_1_states & child_2_states
            if len(intersec) == 0:
                # Intersection is empty -> mutation occurred
                site_dict[parent_node] = child_1_states | child_2_states
                s_site += 1
            else:
                # Intersection is not empty -> no mutation
                site_dict[parent_node] = intersec

        elif len(node_indices) == 1:
            # Recombination structure (single child)
            children_node = arg_site.edge[node_indices, 1].astype(int)
            site_dict[parent_node] = site_dict[children_node[0]]

    s_vec[site_loc] = s_site

# Calculate homoplasy index
m_sum = np.sum(m_vec)
s_sum = np.sum(s_vec)

In [19]:
1 - m_sum / s_sum

np.float64(0.046875)

## Convert to SimBac local tree

In [25]:
node_site

array([[ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True],
       ...,
       [ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True]], shape=(283, 2000))

In [26]:
node_site = np.loadtxt("../../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)
node_site

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(10, 100000))

In [98]:
start_pos = 1
end_pos = start_pos + 10000 - 1
start_pos, end_pos

(1, 10000)

In [86]:
file_path = "../../data/SimBac/local_tree.nwk" # Replace with your actual file name

blocks = load_local_trees(file_path)

print(f"\nSuccessfully loaded {len(blocks)} local trees.")

# Let's inspect the first few blocks to see where they map on the genome
print("\n--- Genome Map Overview ---")
current_position = 1

for i, (span, tree) in enumerate(blocks[:20]): # Just looking at the first 20
    end_position = current_position + span - 1
    print(f"Block {i+1}: Sites {current_position:05d} to {end_position:05d} (Length: {span}bp) - Tree has {len(tree.get_terminals())} leaves")
    current_position += span

Parsing local trees...

Successfully loaded 10618 local trees.

--- Genome Map Overview ---
Block 1: Sites 00001 to 00025 (Length: 25bp) - Tree has 10 leaves
Block 2: Sites 00026 to 00060 (Length: 35bp) - Tree has 10 leaves
Block 3: Sites 00061 to 00088 (Length: 28bp) - Tree has 10 leaves
Block 4: Sites 00089 to 00119 (Length: 31bp) - Tree has 10 leaves
Block 5: Sites 00120 to 00123 (Length: 4bp) - Tree has 10 leaves
Block 6: Sites 00124 to 00125 (Length: 2bp) - Tree has 10 leaves
Block 7: Sites 00126 to 00136 (Length: 11bp) - Tree has 10 leaves
Block 8: Sites 00137 to 00139 (Length: 3bp) - Tree has 10 leaves
Block 9: Sites 00140 to 00149 (Length: 10bp) - Tree has 10 leaves
Block 10: Sites 00150 to 00152 (Length: 3bp) - Tree has 10 leaves
Block 11: Sites 00153 to 00153 (Length: 1bp) - Tree has 10 leaves
Block 12: Sites 00154 to 00155 (Length: 2bp) - Tree has 10 leaves
Block 13: Sites 00156 to 00165 (Length: 10bp) - Tree has 10 leaves
Block 14: Sites 00166 to 00172 (Length: 7bp) - Tree 

In [100]:
n_leaf = node_site.shape[0]
n_site = 10000
m_vec = np.zeros(n_site, dtype=int)
s_vec = np.zeros(n_site, dtype=int)
n_leaf, n_site, m_vec, s_vec

(10,
 10000,
 array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)))

Require:
- arg_site.edge
- node_vec
- leaf_states

In [101]:
local_tree, start, end = get_tree_at_position(blocks, start_pos)

In [102]:
Phylo.draw_ascii(local_tree), start, end

                                                    ______________________ 4
  _________________________________________________|
 |                                                 |                  ____ 2
 |                                                 |_________________|
 |                                                                   |____ 8
 |
_|                                                       _________________ 9
 |                                                      |
 |                                             _________|__________________ 5
 |                                            |         |
 |                                            |         |                , 6
 |                                            |         |   _____________|
 |____________________________________________|         |__|             | 7
                                              |            |
                                              |            |______________ 3
    

(None, 1, 25)

In [103]:
local_edge, local_node_height = newick_to_tree(local_tree)
local_edge, local_node_height

(array([[1.1000000e+01, 9.0000000e+00, 8.5366000e-02],
        [1.1000000e+01, 3.0000000e+00, 8.5366000e-02],
        [1.2000000e+01, 1.1000000e+01, 2.6743300e-01],
        [1.2000000e+01, 5.0000000e+00, 3.5279900e-01],
        [1.3000000e+01, 8.0000000e+00, 2.9385610e-02],
        [1.3000000e+01, 7.0000000e+00, 2.9385610e-02],
        [1.4000000e+01, 4.0000000e+00, 2.3634561e-01],
        [1.4000000e+01, 1.3000000e+01, 2.0696000e-01],
        [1.5000000e+01, 6.0000000e+00, 2.7620011e-01],
        [1.5000000e+01, 1.4000000e+01, 3.9854500e-02],
        [1.6000000e+01, 1.0000000e+01, 2.8275000e-01],
        [1.6000000e+01, 1.5000000e+01, 6.5498900e-03],
        [1.7000000e+01, 1.0000000e+00, 4.9792000e-02],
        [1.7000000e+01, 2.0000000e+00, 4.9792000e-02],
        [1.8000000e+01, 1.7000000e+01, 3.7510300e-01],
        [1.8000000e+01, 1.6000000e+01, 1.4214500e-01],
        [1.9000000e+01, 1.2000000e+01, 7.4856700e-01],
        [1.9000000e+01, 1.8000000e+01, 6.7647100e-01]]),
 array([

In [104]:
local_pos = start_pos
local_pos

1

In [105]:
for i in range(n_site):
    # Get local tree at this site
    if not start <= local_pos <= end:
        local_tree, start, end = get_tree_at_position(blocks, local_pos)
        local_edge, local_node_height = newick_to_tree(local_tree)

    node_vec = np.arange(1, 2*n_leaf)

    # Compute minimum possible changes
    leaf_states = node_site[:, local_pos - 1]  # Convert to 0-indexed
    if np.any(leaf_states) and not np.all(leaf_states):
        m_vec[i] = 1

    # Compute actual changes using Fitch algorithm
    s_site = 0
    site_dict = {}

    # Initialize leaf nodes
    site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

    # Process internal nodes
    for i in range(n_leaf, len(node_vec)):
        parent_node = node_vec[i]
        node_indices = np.where(local_edge[:, 0] == parent_node)[0]

        # Coalescent structure (two children)
        children_node = local_edge[node_indices, 1].astype(int)
        child_1_states = site_dict[children_node[0]]
        child_2_states = site_dict[children_node[1]]

        intersec = child_1_states & child_2_states
        if len(intersec) == 0:
            # Intersection is empty -> mutation occurred
            site_dict[parent_node] = child_1_states | child_2_states
            s_site += 1
        else:
            # Intersection is not empty -> no mutation
            site_dict[parent_node] = intersec

    s_vec[i] = s_site
    local_pos += 1

# Calculate homoplasy index
m_sum = np.sum(m_vec)
s_sum = np.sum(s_vec)

In [107]:
s_vec, s_sum

(array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)), np.int64(0))

In [108]:
1 - m_sum / s_sum

C:\Users\u2008181\AppData\Local\Temp\ipykernel_7240\1426461223.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  1 - m_sum / s_sum


np.float64(-inf)